# Lab: Building an Interactive Dashboard

In the lecture, we built a dashboard one piece at a time. In this lab, you will bring together the analysis workflow, Plotly figures, and Dash components to create an interactive dashboard using a dataset of your choosing.

Your dashboard should help a specific audience answer a focused question. The goal is not to fit every possible chart onto one page. Select a small set of related evidence, organize it clearly, and give the user one useful way to interact with it.



## Learning goals

By the end of this lab, you should be able to:

- use the reproducible analysis workflow to plan a dashboard;
- prepare and check data before displaying it;
- create Plotly figures that answer related questions;
- organize text, controls, and figures in a Dash layout;
- connect an interactive control to figures with a callback; and
- explain the conclusion and limitations of a dashboard.



## Using AI tools responsibly

You may use an AI tool to ask questions, debug code, or receive feedback. You remain responsible for understanding every part of the submitted work, testing the dashboard, citing the dataset, and checking that the evidence supports your conclusion. Do not submit private, confidential, or personally identifying data to an AI system.



## Assignment requirements

Choose a dataset that supports a meaningful comparison. It should contain:

- enough observations to reveal a pattern rather than a few isolated cases;
- at least one categorical feature that can be used as an interactive filter or comparison;
- at least two additional useful features, which may be categorical, quantitative, temporal, spatial, or text-derived; and
- enough documentation for you to explain what the records and selected features represent.

The completed dashboard must contain:

- a descriptive title and a short explanation of its purpose;
- at least two related Plotly figures;
- at least one dropdown menu or radio-button control;
- at least one callback that updates both figures;
- human-readable titles, labels, categories, and colors;
- stable axes or category orders when changing them would make comparisons misleading; and
- a brief conclusion and limitation displayed on the page.



# Step 1: Question



## 1. Identify the audience

A dashboard should be designed for someone. Identify a realistic audience and explain what decision, judgment, or understanding the dashboard could support.



**Who is the intended audience, and why would this evidence be useful to them?**

My audience is restaurant servers and managers. The dashboard helps them explore how tip amounts relate to bill amounts and party sizes during lunch and dinner.

## 2. State the research question

Write one focused question that your two figures will help answer. The question should name the main features or groups being compared.



**What question will the dashboard answer?**

How do tip amounts relate to the total bill and party size, and how do these patterns compare between lunch and dinner?

## 3. Make a prediction



**What pattern do you expect to find, and what informed that prediction?**

I expect larger bills to generally come with larger tips. I also expect bigger parties to leave larger total tips because they usually order more food. This does not necessarily mean they leave a higher tip percentage.

# Step 2: Data



## 4. Import the libraries

Import Pandas, Plotly Express, and the Dash objects used in the lecture: `Dash`, `dcc`, `html`, `Input`, and `Output`.



In [1]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

## 5. Document the dataset

Give the dataset title, its creator or publisher, a working link or citation, the time period represented, and any relevant collection or sampling information.



**Where did the data come from, and what does its documentation tell you about how it was collected?**

I used the Tips dataset distributed with Plotly. The documentation says one waiter recorded 244 tips over a few months at one restaurant. The data was reported in Bryant and Smith's 1995 book, Practical Data Analysis: Case Studies in Business Statistics. The exact collection dates are not provided in this documentation. This is a small historical dataset, not a random sample of all restaurants.

Sources:

- [Tips dataset documentation](https://rdrr.io/cran/reshape2/man/tips.html)
- [Tips data file in Plotly’s official repository](https://github.com/plotly/datasets/blob/master/tips.csv)

## 6. Read the data

Read the dataset into a Pandas DataFrame. Give the DataFrame a short, descriptive name.



In [2]:
# Get the restaurant dataset included with Plotly
tips = px.data.tips()

# Save the data file to include with the assignment
tips.to_csv("tips.csv", index=False)

# Read the saved file
tips = pd.read_csv("tips.csv")

print("Data loaded:", len(tips), "rows")

Data loaded: 244 rows


## 7. Preview and inspect the DataFrame

Display the first five rows, last five rows, stored data types, and missing-value counts.



In [3]:
display(tips.head())
display(tips.tail())
display(tips.dtypes)
tips.info()
display(tips.isnull().sum())

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


,total_bill,tip,sex,smoker,day,time,size
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2
243,18.78,3.00,Female,No,Thur,Dinner,2


total_bill    float64
tip           float64
sex               str
smoker            str
day               str
time              str
size            int64
dtype: object

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   total_bill  244 non-null    float64
 1   tip         244 non-null    float64
 2   sex         244 non-null    str    
 3   smoker      244 non-null    str    
 4   day         244 non-null    str    
 5   time        244 non-null    str    
 6   size        244 non-null    int64  
dtypes: float64(2), int64(1), str(4)
memory usage: 17.3 KB


total_bill    0
tip           0
sex           0
smoker        0
day           0
time          0
size          0
dtype: int64

**Complete the table for every feature you expect to use. Add or remove rows as needed.**

| Feature | What it represents | Data storage type | Feature type | Missing values | Dashboard role |
|---|---|---|---|---:|---|
| total_bill | Bill amount in dollars | float64 | Quantitative | 0 | Scatter plot horizontal axis |
| tip | Tip amount in dollars | float64 | Quantitative | 0 | Scatter plot vertical axis and average for bars |
| time | Lunch or dinner | str | Categorical | 0 | Dropdown and chart colors |
| size | Number of people in the party | int64 | Quantitative, discrete | 0 | Bar chart groups and hover information |



**What does one row of the original dataset represent?**

One row represents one recorded restaurant bill and tip for a party of customers.

# Step 3: Operation



## 8. Create the analysis DataFrame

Select the features needed for the dashboard and use `.copy()` to create a DataFrame named `analysis`. Handle missing values in a way that is appropriate for the question. If values need clearer labels or corrected storage types, make those changes here.



In [4]:
analysis = tips[["total_bill", "tip", "time", "size"]].copy()
analysis = analysis.dropna()
analysis = analysis.rename(columns={"time": "meal", "size": "party_size"})

**How did you handle missing values, and why was that choice appropriate?**

I checked the selected columns and found no missing values. I included dropna() to exclude any rows missing values needed for the charts, but it did not remove any rows from this dataset.

## 9. Create any calculated features

If the dashboard needs a calculated category, date component, percentage, rate, or other calculated feature, create it here. If no calculated feature is needed, leave the code cell blank and explain why.

**What calculated feature or cleaned label did you create, and how will it support the dashboard?**

I renamed time to meal and size to party_size to make the names clearer. I do not need another column for each row. For the bar chart, I calculate the average tip and number of records for each meal and party size.

## 10. Prepare the control options

Create the labels and values for the dropdown or radio-button control. The visible labels should be understandable to someone who has not seen the original column names.



In [5]:
meal_order = ["Lunch", "Dinner"]
meal_colors = {"Lunch": "royalblue", "Dinner": "darkorange"}
control_options = [
    {"label": "Lunch and dinner", "value": "All"},
    {"label": "Lunch", "value": "Lunch"},
    {"label": "Dinner", "value": "Dinner"}
]

# Step 4: Check



## 11. Check the prepared data

Display the first rows of `analysis`, its stored data types, missing-value counts, and appropriate summaries or frequency counts for the dashboard features.



In [6]:
display(analysis.head())
display(analysis.dtypes)
display(analysis.isnull().sum())
display(analysis[["total_bill", "tip", "party_size"]].describe())
display(analysis["meal"].value_counts())
summary = analysis.groupby(["meal", "party_size"], as_index=False).agg(
    average_tip=("tip", "mean"), records=("tip", "size")
)
display(summary.round(2))

,total_bill,tip,meal,party_size
0,16.99,1.01,Dinner,2
1,10.34,1.66,Dinner,3
2,21.01,3.50,Dinner,3
3,23.68,3.31,Dinner,2
4,24.59,3.61,Dinner,4


total_bill    float64
tip           float64
meal              str
party_size      int64
dtype: object

total_bill    0
tip           0
meal          0
party_size    0
dtype: int64

,total_bill,tip,party_size
count,244.000000,244.000000,244.000000
mean,19.785943,2.998279,2.569672
std,8.902412,1.383638,0.951100
min,3.070000,1.000000,1.000000
25%,13.347500,2.000000,2.000000
50%,17.795000,2.900000,2.000000
75%,24.127500,3.562500,3.000000
max,50.810000,10.000000,6.000000


meal
Dinner    176
Lunch      68
Name: count, dtype: int64

,meal,party_size,average_tip,records
0,Dinner,1,1.00,2
1,Dinner,2,2.66,104
2,Dinner,3,3.49,33
3,Dinner,4,4.12,32
4,Dinner,5,3.78,4
5,Dinner,6,5.00,1
6,Lunch,1,1.88,2
7,Lunch,2,2.42,52
8,Lunch,3,2.75,5
9,Lunch,4,4.22,5


## 12. Check the interactive groups

Calculate the number of observations available for every control option. Make sure each option returns data and that unexpectedly small groups are understood before building the callback.



In [7]:
for option in control_options:
    selected = option["value"]
    subset = analysis if selected == "All" else analysis[analysis["meal"] == selected]
    print(option["label"], ":", len(subset), "records")
    assert len(subset) > 0

display(pd.crosstab(analysis["party_size"], analysis["meal"]))

Lunch and dinner : 244 records
Lunch : 68 records
Dinner : 176 records


meal,Dinner,Lunch
party_size,,
1,2,2
2,104,52
3,33,5
4,32,5
5,4,1
6,1,3


**What did these checks establish about the data that will appear in the dashboard?**

All three dropdown choices have data. There are 244 records in total, with 68 lunch records and 176 dinner records. The selected columns have no missing values, and party sizes range from one to six people. Some groups are very small. For example, lunch has only one five-person party, and dinner has only one six-person party, so their averages should be treated carefully.

# Step 5: Evidence



## 13. Create the first static Plotly figure

Build the first figure before placing it in Dash. Give it a descriptive title and human-readable axis and legend labels.



In [8]:
scatter = px.scatter(
    analysis, x="total_bill", y="tip", color="meal",
    hover_data=["party_size"],
    labels={"total_bill": "Total bill ($)", "tip": "Tip ($)",
            "meal": "Meal", "party_size": "Party size"},
    color_discrete_map=meal_colors, category_orders={"meal": meal_order},
    title="Bill Amount and Tip — Lunch and Dinner",
    range_x=[0, 55], range_y=[0, 11], template="plotly_white"
)
scatter.show()

**What evidence should a viewer obtain from the first figure?**

The scatter plot shows whether larger bills tend to come with larger tips. Each dot represents one recorded bill and tip, and the colors identify lunch and dinner.

## 14. Create the second static Plotly figure

The second figure should add different but related evidence rather than repeat the first figure in another style.



In [9]:
bars = px.bar(
    summary, x="party_size", y="average_tip", color="meal", barmode="group",
    hover_data={"average_tip": ":.2f", "records": True},
    labels={"party_size": "Party size (people)", "average_tip": "Average tip ($)",
            "meal": "Meal", "records": "Number of records"},
    color_discrete_map=meal_colors, category_orders={"meal": meal_order},
    title="Average Tip by Party Size — Lunch and Dinner",
    range_x=[0.5, 6.5], range_y=[0, 6], template="plotly_white"
)
bars.update_xaxes(dtick=1)
bars.show()

**What does the second figure add to the first figure's evidence?**

The bar chart compares average tips for different party sizes. It helps viewers see how total tips differ between smaller and larger parties instead of only looking at individual bills. Hovering also shows how many records support each average.

## 15. Plan the dashboard



**Complete the dashboard plan before writing the app. Add rows if needed.**

| Component | What it displays or controls | Why it belongs on the dashboard |
|---|---|---|
| Title and introduction | Restaurant bills, tips, and the dashboard's purpose | Explains the topic |
| Interactive control | Lunch, dinner, or both | Lets the viewer focus on a meal |
| First figure | Bill amount versus tip | Shows individual bills and tips |
| Second figure | Average tip by party size | Adds a comparison between party sizes |
| Conclusion and limitation | Main pattern and limits of the data | Helps viewers avoid overgeneralizing |



## 16. Create reusable figure code

Write a function that accepts the selected control value, filters or summarizes `analysis`, creates both Plotly figures, and returns the figures in a consistent order. Fix axis ranges or category orders when they need to remain stable across selections.



In [10]:
def make_figures(selected_meal):
    if selected_meal == "All":
        filtered = analysis
        selection_title = "Lunch and Dinner"
    else:
        filtered = analysis[analysis["meal"] == selected_meal]
        selection_title = selected_meal

    averages = filtered.groupby(["meal", "party_size"], as_index=False).agg(
        average_tip=("tip", "mean"), records=("tip", "size")
    )

    first = px.scatter(
        filtered, x="total_bill", y="tip", color="meal",
        hover_data=["party_size"],
        labels={"total_bill": "Total bill ($)", "tip": "Tip ($)",
                "meal": "Meal", "party_size": "Party size"},
        color_discrete_map=meal_colors, category_orders={"meal": meal_order},
        title=f"Bill Amount and Tip — {selection_title}",
        range_x=[0, 55], range_y=[0, 11], template="plotly_white"
    )
    second = px.bar(
        averages, x="party_size", y="average_tip", color="meal", barmode="group",
        hover_data={"average_tip": ":.2f", "records": True},
        labels={"party_size": "Party size (people)", "average_tip": "Average tip ($)",
                "meal": "Meal", "records": "Number of records"},
        color_discrete_map=meal_colors, category_orders={"meal": meal_order},
        title=f"Average Tip by Party Size — {selection_title}",
        range_x=[0.5, 6.5], range_y=[0, 6], template="plotly_white"
    )
    second.update_xaxes(dtick=1)
    for figure in (first, second):
        figure.update_layout(height=420)
    return first, second

## 17. Build the layout

Create a Dash app and its layout. Include the title, purpose statement, interactive control, two empty `dcc.Graph` components with unique IDs, and short conclusion and limitation statements.



In [11]:
app = Dash(__name__)

app.layout = html.Div([
    html.H1("Restaurant Bills and Tips"),

    html.P(
        "Explore how tip amounts relate to bill amounts and party sizes "
        "at lunch and dinner."
    ),

    html.Label("Choose a meal:", htmlFor="meal-choice"),

    dcc.Dropdown(
        id="meal-choice",
        options=control_options,
        value="All",
        clearable=False
    ),

    dcc.Graph(id="bill-tip-chart"),
    dcc.Graph(id="party-tip-chart"),

    html.H3("Conclusion from the full dataset"),

    html.P(
        "Larger bills generally came with larger tips. Four-person parties "
        "averaged $4.22 at lunch and $4.12 at dinner, compared with $2.42 "
        "and $2.66 for two-person parties. Bigger parties did not always "
        "leave higher tips."
    ),

    html.H3("Limitation"),

    html.P(
        "These historical records come from one waiter at one restaurant. "
        "Some party-size groups contain only one or two records. "
        "The results may not represent other restaurants or current "
        "tipping habits. Tip amounts are dollars, not percentages."
    ),

    html.P([
        "Source: ",
        html.A(
            "Tips dataset documentation",
            href="https://rdrr.io/cran/reshape2/man/tips.html"
        )
    ])
], style={
    "maxWidth": "1000px",
    "margin": "auto",
    "padding": "20px",
    "fontFamily": "Arial"
})

## 18. Connect the callback

Connect the control's `value` to the `figure` property of both graphs. The callback function should call the reusable figure function and return its two figures in the same order as the two outputs.



In [12]:
@app.callback(
    Output("bill-tip-chart", "figure"),
    Output("party-tip-chart", "figure"),
    Input("meal-choice", "value")
)
def update_dashboard(selected_meal):
    return make_figures(selected_meal)

## 19. Check the callback as a Python function

Call the callback or reusable figure function with at least two valid control values. Confirm that each call returns two Plotly figures without an error.



In [13]:
from plotly.graph_objects import Figure

for selection in ["All", "Lunch", "Dinner"]:
    first, second = update_dashboard(selection)
    assert isinstance(first, Figure)
    assert isinstance(second, Figure)
    assert len(first.data) > 0 and len(second.data) > 0
    print(selection, "— both figures returned successfully")

All — both figures returned successfully
Lunch — both figures returned successfully
Dinner — both figures returned successfully


## 20. Run the dashboard

Run the app in a browser tab. Use a port that is not already occupied by another app, and stop the app before restarting the same code.



In [16]:
app.run(jupyter_mode="tab", port=8055, debug=False)

Dash app running on http://127.0.0.1:8055/


<IPython.core.display.Javascript object>

**After testing every control option, what did you revise or confirm about the dashboard's behavior?**

I confirmed that lunch, dinner, and the combined option all update both charts. The chart titles change with the selection, while the axis ranges and meal colors stay the same.

**Which design choice most improves the dashboard's readability or usefulness?**

Keeping the same axis ranges and meal colors makes the charts easier to compare. Lunch stays blue and dinner stays orange, so the meaning of the colors does not change.

# Step 6: Conclusion



**State a concise conclusion that directly answers the research question and refers to evidence visible in the dashboard.**

Larger bills generally came with larger tips. Four-person parties averaged 4.22 dollars at lunch and 4.12 dollars at dinner, compared with 2.42 dollars at lunch and 2.66 dollars at dinner for two-person parties. However, average tips did not increase with every increase in party size. Some groups also have very few records, so these patterns should be interpreted carefully.

**How does the interactive control help the audience examine the evidence?**

The dropdown lets viewers look at lunch, dinner, or both. Since both charts update together, viewers can explore the relationship between bills and tips and compare party sizes for the same meal.

# Step 7: Limitation



**State one important limitation on how the dashboard's evidence should be interpreted.**

The data comes from one waiter at one restaurant over a few months. The results may not represent other restaurants or current tipping habits, especially because some party-size groups have very few records.

## AI-use reflection



**Describe any AI assistance you used. Identify one suggestion you verified, revised, or rejected. If you did not use AI, state that.**

I used AI for help troubleshooting errors and reviewing parts of my dashboard. One suggestion I checked was loading the Tips dataset from Plotly and saving it as a CSV when the notebook could not find the file. After making that change, I confirmed that 244 rows loaded and tested all three dropdown options to check that both charts updated.

## Submission checklist

Before submitting, confirm that:

- [x] every prompt has one complete response;
- [x] all notebook cells run in order without an error;
- [x] the dashboard opens and every control option works;
- [x] the two figures use readable titles, labels, categories, and colors;
- [x] comparable views retain stable axes or category orders when needed;
- [x] the dataset source is cited and the required data file accompanies the submission;
- [x] the notebook and dashboard code are organized and readable;
- [x] the conclusion and limitation are supported by the displayed evidence; and
- [x] the work is submitted by the deadline.

